In [1]:
# Import all necessary libraries
import numpy as np 
import random
import matplotlib.pyplot as plt
import math
import pickle
import os
import pandas as pd
import random
import time
import sys
from pathlib import Path

# Add src directory to path for imports
sys.path.insert(0, str(Path(__file__).parent.parent / 'src')) if '__file__' in globals() else sys.path.insert(0, str(Path.cwd().parent / 'src')) if Path.cwd().name == 'notebooks' else sys.path.insert(0, 'src')

plt.rcParams['figure.figsize'] = [10, 7]
#import mpl_scatter_density # adds projection='scatter_density'

import gc

# Import from refactored modules
from nMELTS.data.parser import BigMetaTable
import nMELTS.data.parser as BMT  # Keep BMT alias for backward compatibility
from nMELTS.config import *  # Import all constants 
from nMELTS.config.settings import external_base, internal_data_dir, external_data_dir  
from nMELTS.utils.file_utils import delete_files_with_keyword, move_files_with_extension
from nMELTS.config.indexer import DatasetIndexer

# Here set where your inputs and outputs are. Internal dir must be the same structure as the data on the external dir


In [2]:
"""Define all labeling systems / dictionaries for this module"""

# Old Filters for deep_filter function
Oxide_Lower_Bounds = [['melts-liquid', 'SiO2', 37],
                      ['nepheline', 'Na2O', 12.5],
                      ['garnet', 'FeO', 5],
                      ['k-feldspar', 'K2O', 8],
                      ['clinopyroxene', 'CaO', 7],
                      ['orthopyroxene', 'MgO', 13]]

easy_build_oxide_upper_bounds = {'melts-liquid': [ ['NiO', 1], ['MgO', 50], ['TiO2', 6], ['MnO', 1]],
                                 'rhm-oxide': [['MnO', 7]],
                                 'nepheline': [['K2O', 11]],
                                 'k-feldspar': [['CaO', 1]],
                                 'plagioclase': [['K2O', 3]],
                                 'clinopyroxene': [ ['TiO2', 6], ['Al2O3', 13], ['Na2O', 4], ['Fe2O3', 3] ],
                                 'orthopyroxene': [ ['TiO2', 2], ['Al2O3', 7], ['Na2O', 0.2], ['Fe2O3', 3], ['CaO', 4] ],
                                 'olivine': [ ['CaO', 0.6], ['NiO', 2], ['MnO', 1] ]
                                }
                                      
Oxide_Upper_Bounds = []

for phase, boundlist in easy_build_oxide_upper_bounds.items():
    for ox, bound in boundlist:
        Oxide_Upper_Bounds.append([phase, ox, bound])
print(Oxide_Upper_Bounds)    

[['melts-liquid', 'NiO', 1], ['melts-liquid', 'MgO', 50], ['melts-liquid', 'TiO2', 6], ['melts-liquid', 'MnO', 1], ['rhm-oxide', 'MnO', 7], ['nepheline', 'K2O', 11], ['k-feldspar', 'CaO', 1], ['plagioclase', 'K2O', 3], ['clinopyroxene', 'TiO2', 6], ['clinopyroxene', 'Al2O3', 13], ['clinopyroxene', 'Na2O', 4], ['clinopyroxene', 'Fe2O3', 3], ['orthopyroxene', 'TiO2', 2], ['orthopyroxene', 'Al2O3', 7], ['orthopyroxene', 'Na2O', 0.2], ['orthopyroxene', 'Fe2O3', 3], ['orthopyroxene', 'CaO', 4], ['olivine', 'CaO', 0.6], ['olivine', 'NiO', 2], ['olivine', 'MnO', 1]]


In [4]:
internal_data_dir('')

WindowsPath('C:/Git_Repositories/nMELTS/src/nMELTS/Workspace/Datasets')

In [ ]:
# Batch Melting data processing workflow

MELTSModel= '102'
Date = 'Nov9'
preprocessed = False
subset = True

#internal_dir = 'Workspace/{MELTSModel}Datasets/' Get from Emulatorlibrary
#external_dir = external_base + internal_dir(MELTSModel)

# Make Directories if not present
if not os.path.exists('Workspace/'):
    os.makedirs('Workspace/')
if not os.path.exists(external_base+'Workspace/'):
    os.makedirs(external_base+'Workspace/')
if not os.path.exists(internal_data_dir(MELTSModel)):
    os.makedirs(internal_data_dir(MELTSModel))
if not os.path.exists(external_data_dir(MELTSModel)):
    os.makedirs(external_data_dir(MELTSModel))

ValidName = f"{internal_data_dir(MELTSModel)}MELTS{MELTSModel}_Validset{Date}BatchCooling"
TestName = f"{internal_data_dir(MELTSModel)}MELTS{MELTSModel}_Testset{Date}BatchCooling"
TrainName = f'{internal_data_dir(MELTSModel)}MELTS{MELTSModel}_Trainset{Date}BatchCooling'

if subset: # Change name to grab processed .npy and .txt files
    ValidName += '_subset'
    TestName += '_subset'
    TrainName += '_subset'

if preprocessed: # Change name to grab processed .npy and .txt files
    ValidName += '_processed'
    TestName += '_processed'
    TrainName += '_processed'


#time.sleep(3600) #1 Hour Sleep

Oxide_Lower_Bounds = [['k-feldspar', 'K2O', 8],
                      ['clinopyroxene', 'CaO', 7]]

easy_build_oxide_upper_bounds = {'k-feldspar': [['CaO', 1]],
                                 'plagioclase': [['K2O', 8]],
                                 'clinopyroxene': [ ['TiO2', 6], ['Al2O3', 8] ],
                                 'orthopyroxene': [ ['TiO2', 2], ['Al2O3', 8], ['CaO', 4] ],
                                }
                                      
Oxide_Upper_Bounds = []

for phase, boundlist in easy_build_oxide_upper_bounds.items():
    for ox, bound in boundlist:
        Oxide_Upper_Bounds.append([phase, ox, bound])
print(Oxide_Upper_Bounds)    

Component_Upper_Bounds = [] #[ ['plagioclase', 'highsanidine', 0.1] ]


gc.collect()
TrainMELTS = BMT.BigMetaTable(TrainName, read_dir=external_base)
if not preprocessed:
    TrainMELTS.separate_analcime()
    # Subsampling breaks continuity neecsary to blur phase boundaries
    #TrainMELTS.filter_all(generate_binaries=True, thresholds = np.array([6,12,4,4,4,5,10]))
    TrainMELTS.filter_full_metadata()
    TrainMELTS.filter_legal()


    #TrainMELTS.filename = f"{TrainName}Filtered"
    TrainMELTS.resample_rare_phase(MELTS_indices['nepheline']['mass (gm)'], multiplier_bounds = (0.8,1.1), n_resamples=10, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['leucite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=2, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['analcime']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=3, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['olivine']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['muscovite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=3, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['biotite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['k-feldspar']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)


    BMT.balance_lowF(TrainMELTS)
    TrainMELTS.save(name = f"{TrainName}Filtered", save_csv=False) 
    #TrainMELTS.filename = f'{TrainName}Resampled'

TrainMELTS.filename = TrainName
BMT.resampling_to_datasets(TrainMELTS,[[1,1],[.8,1],[0.5,1]]) #[[0.9,1]])

del TrainMELTS.table
del TrainMELTS
gc.collect()

# Clear intermediate memory maps
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)

# Rename processed data and move it to external directory
if not preprocessed: 
    os.rename(TrainName + 'Filtered.npy', TrainName + '_processed.npy')
    os.rename(TrainName + 'Filtered.txt', TrainName + '_processed.txt')


BMT.deep_filter(f'{TrainName}', Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)

# Move all .npy data products
"""move_files_with_extension(extension = '.npy', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)"""



#train_plot_directory = 'Training_Dataset_July28Intensive'

#BMT.make_harkers(TrainMELTS, train_plot_directory+'/')
#BMT.make_Tplots(TrainMELTS, train_plot_directory+'/') 
#BMT.F_phase_plots(partition_directory=train_plot_directory, filename=TrainMELTS.filename)

#time.sleep(4200)
""


ValidMELTS = BMT.BigMetaTable(ValidName, read_dir=external_base)
if not preprocessed:
    ValidMELTS.separate_analcime()
    #ValidMELTS.filter_all(generate_binaries=True, thresholds = np.array([6,12,4,4,4,5,10]))
    ValidMELTS.filter_full_metadata()
    ValidMELTS.filter_legal()
    ValidMELTS, TestMELTS = ValidMELTS.split(0.30)



    #TestMELTS.filename = f"{TestName}Filtered"


    TestMELTS.resample_rare_phase(component_indices['nepheline']['mass (gm)'], multiplier_bounds = (0.99,1.01), n_resamples=10, overwrite=True)
    TestMELTS.resample_rare_phase(component_indices['leucite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=2, overwrite =True)
    TestMELTS.resample_rare_phase(component_indices['analcime']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=3, overwrite =True)
    TestMELTS.resample_rare_phase(component_indices['olivine']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)
    TestMELTS.resample_rare_phase(component_indices['muscovite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=3, overwrite =True)
    TestMELTS.resample_rare_phase(component_indices['biotite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)
    TestMELTS.resample_rare_phase(component_indices['k-feldspar']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)

    print("################ After resampling ####################")

    BMT.balance_lowF(TestMELTS)
    BMT.balance_lowF(ValidMELTS)
    TestMELTS.save(name = f"{TestName}Filtered", save_csv=False) 
    ValidMELTS.save(name = f"{ValidName}Filtered", save_csv=False) 

else:
    TestMELTS = BMT.BigMetaTable(TestName) # If processed, this file will exist already

TestMELTS.filename = TestName
ValidMELTS.filename = ValidName

BMT.resampling_to_datasets(TestMELTS)
BMT.resampling_to_datasets(ValidMELTS)

valid_plot_directory = f"Plots/{ValidName}Harkers"

BMT.make_harkers(ValidMELTS, valid_plot_directory+'/')
BMT.make_Tplots(ValidMELTS, valid_plot_directory+'/') 

del TestMELTS.table, ValidMELTS.table
del TestMELTS, ValidMELTS
gc.collect()

delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)



    #move_file(TrainName + '_processed.npy', dst_dir = external_directory + TrainName + '_processed.npy', overwrite=True)
    #move_file(TrainName + '_processed.txt', dst_dir = external_directory + TrainName + '_processed.txt', overwrite=True)

# Move all .npy data products


BMT.deep_filter(TestName, Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)
BMT.deep_filter(ValidName, Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)

# Rename processed data and move it to external directory
if not preprocessed: 
    os.rename(ValidName + 'Filtered.npy', ValidName + '_processed.npy')
    os.rename(ValidName + 'Filtered.txt', ValidName + '_processed.txt')
    os.rename(TestName + 'Filtered.npy', TestName + '_processed.npy')
    os.rename(TestName + 'Filtered.txt', TestName + '_processed.txt')

#BMT.F_phase_plots(partition_directory=valid_plot_directory, filename=ValidMELTS.filename)
"""move_files_with_extension(extension = '.npy', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)
move_files_with_extension(extension = '.txt', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)
move_files_with_extension(extension = '.csv', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)"""

#

In [ ]:
TrainMELTS.table.shape

In [ ]:
move_files_with_extension(extension = '.npy', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)
move_files_with_extension(extension = '.txt', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)
move_files_with_extension(extension = '.csv', dst_dir = external_dir(MELTSModel), src_dir = internal_dir(MELTSModel), overwrite=True)

In [ ]:
# Fractional Melting/Crystalization data processing workflow


MELTSModel= '102'
Date = 'Nov9'
preprocessed = False
subset = False # (If using databases truncated by code below, just changes name of path accordingly.)

# Make Directories if not present
if not os.path.exists('Workspace/'):
    os.makedirs('Workspace/')
if not os.path.exists(external_base+'Workspace/'):
    os.makedirs(external_base+'Workspace/')
if not os.path.exists(internal_dir(MELTSModel)):
    os.makedirs(internal_dir(MELTSModel))
if not os.path.exists(external_dir(MELTSModel)):
    os.makedirs(external_dir(MELTSModel))

ValidName = f"{internal_dir(MELTSModel)}MELTS{MELTSModel}_Validset{Date}FxCrystCooling"
TestName = f"{internal_dir(MELTSModel)}MELTS{MELTSModel}_Testset{Date}FxCrystCooling"
TrainName = f'{internal_dir(MELTSModel)}MELTS{MELTSModel}_Trainset{Date}FxCrystCooling'

if subset: # Change name to grab processed .npy and .txt files
    ValidName += '_subset'
    TestName += '_subset'
    TrainName += '_subset'

if preprocessed: # Change name to grab processed .npy and .txt files
    ValidName += '_processed'
    TestName += '_processed'
    TrainName += '_processed'

#time.sleep(2700) #1 Hour Sleep

Oxide_Lower_Bounds = [['k-feldspar', 'K2O', 8],
                      ['clinopyroxene', 'CaO', 7]]

easy_build_oxide_upper_bounds = {'k-feldspar': [['CaO', 1]],
                                 'plagioclase': [['K2O', 8]],
                                 'clinopyroxene': [ ['TiO2', 6], ['Al2O3', 8] ],
                                 'orthopyroxene': [ ['TiO2', 2], ['Al2O3', 8], ['CaO', 4] ],
                                }
                                      
Oxide_Upper_Bounds = []

for phase, boundlist in easy_build_oxide_upper_bounds.items():
    for ox, bound in boundlist:
        Oxide_Upper_Bounds.append([phase, ox, bound])
print(Oxide_Upper_Bounds)    

Component_Upper_Bounds = []#[ ['plagioclase', 'highsanidine', 0.1] ]


gc.collect()
TrainMELTS = BMT.BigMetaTable(TrainName, read_dir=external_base)
if not preprocessed:
    TrainMELTS.separate_analcime()
    # Subsampling breaks continuity neecsary to blur phase boundaries
    #TrainMELTS.filter_all(generate_binaries=True, thresholds = np.array([6,12,4,4,4,5,10]))
    TrainMELTS.filter_full_metadata()
    TrainMELTS.filter_legal()


    #TrainMELTS.filename = f"{TrainName}Filtered"
    TrainMELTS.resample_rare_phase(MELTS_indices['nepheline']['mass (gm)'], multiplier_bounds = (0.8,1.1), n_resamples=10, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['leucite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=2, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['analcime']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=3, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['olivine']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['muscovite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=3, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['biotite']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)
    TrainMELTS.resample_rare_phase(MELTS_indices['k-feldspar']['mass (gm)'], multiplier_bounds=(0.8,1.1), n_resamples=1, overwrite=True)


    BMT.balance_Superliquidus_fxtal(TrainMELTS)
    TrainMELTS.save(name = f"{TrainName}Filtered", save_csv=False) 
    

    #TrainMELTS.filename = f'{TrainName}Resampled'"""

TrainMELTS.filename = TrainName
BMT.resampling_to_datasets(TrainMELTS, [[1,1],[.8,1],[0.5,1]])#[[0.9,1]])#[[1,1],[0.5,1]])

del TrainMELTS.table
del TrainMELTS
gc.collect()

# Clear intermediate memory maps
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)

# Rename processed data and move it to external directory
if not preprocessed: 
    os.rename(TrainName + 'Filtered.npy', TrainName + '_processed.npy')
    os.rename(TrainName + 'Filtered.txt', TrainName + '_processed.txt')


BMT.deep_filter(f'{TrainName}', Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)

delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)

# Move all .npy data products
move_files_with_extension(extension = '.npy', dst_dir = external_dir(MELTSModel)+'/', src_dir = internal_dir(MELTSModel)+'/', overwrite=True)
move_files_with_extension(extension = '.txt', dst_dir = external_dir(MELTSModel)+'/', src_dir = internal_dir(MELTSModel)+'/', overwrite=True)


#train_plot_directory = 'Training_Dataset_July28Intensive'

#BMT.make_harkers(TrainMELTS, train_plot_directory+'/')
#BMT.make_Tplots(TrainMELTS, train_plot_directory+'/') 
#BMT.F_phase_plots(partition_directory=train_plot_directory, filename=TrainMELTS.filename)








ValidMELTS = BMT.BigMetaTable(ValidName, read_dir=external_base)
if not preprocessed:
    ValidMELTS.separate_analcime()
    #ValidMELTS.filter_all(generate_binaries=True, thresholds = np.array([6,12,4,4,4,5,10]))
    ValidMELTS.filter_full_metadata()
    ValidMELTS.filter_legal()
    ValidMELTS, TestMELTS = ValidMELTS.split(0.30)



    #TestMELTS.filename = f"{TestName}Filtered"

    TestMELTS.resample_rare_phase(MELTS_indices['nepheline']['mass (gm)'], multiplier_bounds = (0.99,1.01), n_resamples=10, overwrite=True)
    TestMELTS.resample_rare_phase(MELTS_indices['leucite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=2, overwrite =True)
    TestMELTS.resample_rare_phase(MELTS_indices['analcime']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=3, overwrite =True)
    TestMELTS.resample_rare_phase(MELTS_indices['olivine']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)
    TestMELTS.resample_rare_phase(MELTS_indices['muscovite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=3, overwrite =True)
    TestMELTS.resample_rare_phase(MELTS_indices['biotite']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)
    TestMELTS.resample_rare_phase(MELTS_indices['k-feldspar']['mass (gm)'], multiplier_bounds=(0.99,1.01), n_resamples=1, overwrite =True)

    TestMELTS.save(name = f"{TestName}Filtered", save_csv=False) 
    ValidMELTS.save(name = f"{ValidName}Filtered", save_csv=False) 

    print("################ After resampling ####################")

    BMT.balance_Superliquidus_fxtal(TestMELTS)
    BMT.balance_Superliquidus_fxtal(ValidMELTS)

else:
    TestMELTS = BMT.BigMetaTable(TestName, read_dir=external_base) # If processed, this file will exist already

TestMELTS.filename = TestName
ValidMELTS.filename = ValidName

BMT.resampling_to_datasets(TestMELTS)
BMT.resampling_to_datasets(ValidMELTS)

valid_plot_directory = f"Plots/{ValidName}Harkers"

BMT.make_harkers(ValidMELTS, valid_plot_directory+'/')
BMT.make_Tplots(ValidMELTS, valid_plot_directory+'/') 

del TestMELTS.table, ValidMELTS.table
del TestMELTS, ValidMELTS
gc.collect()

delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)



    #move_file(TrainName + '_processed.npy', dst_dir = external_directory + TrainName + '_processed.npy', overwrite=True)
    #move_file(TrainName + '_processed.txt', dst_dir = external_directory + TrainName + '_processed.txt', overwrite=True)

# Move all .npy data products


BMT.deep_filter(TestName, Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)
BMT.deep_filter(ValidName, Oxide_Lower_Bounds=Oxide_Lower_Bounds, 
            Oxide_Upper_Bounds=Oxide_Upper_Bounds, Component_Upper_Bounds=Component_Upper_Bounds)

delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'working', dry_run=False)
delete_files_with_keyword(internal_dir(MELTSModel), keyword = 'temp', dry_run=False)

# Rename processed data and move it to external directory
if not preprocessed: 
    os.rename(ValidName + 'Filtered.npy', ValidName + '_processed.npy')
    os.rename(ValidName + 'Filtered.txt', ValidName + '_processed.txt')
    os.rename(TestName + 'Filtered.npy', TestName + '_processed.npy')
    os.rename(TestName + 'Filtered.txt', TestName + '_processed.txt')

#BMT.F_phase_plots(partition_directory=valid_plot_directory, filename=ValidMELTS.filename)

move_files_with_extension(extension = '.npy', dst_dir = external_dir(MELTSModel)+'/', src_dir = internal_dir(MELTSModel)+'/', overwrite=True)
move_files_with_extension(extension = '.txt', dst_dir = external_dir(MELTSModel)+'/', src_dir = internal_dir(MELTSModel)+'/', overwrite=True)


In [ ]:
"""SIMPLE DATA PROCESSING FOR GRID SEARCHES"""

# Batch Melting data processing workflow
for suffix in ['Cr', 'NoCr']:

    for CalcType in ['Batch']:#['Fxtal', 'Batch']:
        for MELTSModel in ['102']:#, '120']:
            Name = f"WSL_MELTS/GTMELTS{MELTSModel}_{suffix}_MORB_{CalcType}_PsuedoSections"

            TrainMELTS = BMT.BigMetaTable(Name)
            TrainMELTS.separate_analcime()
            BMT.resampling_to_datasets(TrainMELTS)
            del TrainMELTS.table
            del TrainMELTS
            gc.collect()



move_files_with_extension(extension = '.npy', dst_dir = 'D:/Workspace/Grid/', src_dir = 'WSL_MELTS/', overwrite=True)
move_files_with_extension(extension = '.txt', dst_dir = 'D:/Workspace/Grid/', src_dir = 'WSL_MELTS/', overwrite=True)
move_files_with_extension(extension = '.csv', dst_dir = 'D:/Workspace/Grid/', src_dir = 'WSL_MELTS/', overwrite=True)




In [ ]:
"""SIMPLE DATA PROCESSING FOR LLDs (Liquid Lines of Descent) """

# Batch Melting data processing workflow
for suffix in ['Cr', 'NoCr']:

    for CalcType in ['Batch']:#['Fxtal', 'Batch']:
        for MELTSModel in ['102']:#, '120']:
            Name = f"WSL_MELTS/GTMELTS{MELTSModel}_{suffix}_MORB_{CalcType}_PsuedoSections_smallOx2"

            TrainMELTS = BMT.BigMetaTable(Name)
            TrainMELTS.separate_analcime()
            BMT.resampling_to_datasets(TrainMELTS)
            del TrainMELTS.table
            del TrainMELTS
            gc.collect()

#move_files_with_extension(extension = '.npy', dst_dir = 'D:/Workspace/Harkers/', src_dir = 'WSL_MELTS/', overwrite=True)
#move_files_with_extension(extension = '.txt', dst_dir = 'D:/Workspace/Harkers/', src_dir = 'WSL_MELTS/', overwrite=True)
#move_files_with_extension(extension = '.csv', dst_dir = 'D:/Workspace/Harkers/', src_dir = 'WSL_MELTS/', overwrite=True)




In [ ]:
"""
ENFORCE 3E6 and 5E5 train and valid sizes respectively
This block is used to create a smaller csv file out of very large initial ones. 
"""
#time.sleep(600)
MELTSModel= '102'
Date = 'Nov9'
calcType = 'Batch'

ValidPath = f"{internal_dir(MELTSModel)}MELTS{MELTSModel}_Validset{Date}{calcType}Cooling"
TrainPath = f"{internal_dir(MELTSModel)}MELTS{MELTSModel}_Trainset{Date}{calcType}Cooling"


valid_inds = BMT.collect_indices(f'{ValidPath}.csv', mass_indices=mass_indices, known_total_rows = None, batch_size=100000, maximum_indices = 5E5, random_state=None, has_header=True)
#valid_inds = np.random.choice(np.arange())
print('Making New csv and txt')
BMT.extract_rows(ValidPath, indices=valid_inds, new_suffix="_subset")
del valid_inds
gc.collect()

train_inds = BMT.collect_indices(f'{TrainPath}.csv', mass_indices=mass_indices, known_total_rows = None, batch_size=100000, maximum_indices = 4E6, random_state=None, has_header=True)
print('Making New csv and txt')
BMT.extract_rows(TrainPath, indices=train_inds, new_suffix="_subset")
del train_inds
gc.collect()

In [ ]:
"""
Spinel Solver. Gradient descent used to find invertible transformation matrix 
that makes all components positive
"""

#Big_table = BMT.BigMetaTable(...Put spinel dataset here)

import torch
import torch.nn as nn
import torch.optim as optim

def solve_masked_A(x, mask=None, lr=0.01, steps=20000, leak=0.001, seed=0):
    """
    Solve for matrix A such that x @ A = g
    using gradient descent on a leaky ReLU loss of -g,
    with some entries fixed to zero but keeping A invertible.

    Args:
        x: (B, N) tensor
        g: (B, M) tensor
        mask: (N, M) binary mask where 1 = learnable, 0 = fixed zero
        lr: learning rate
        steps: number of gradient descent steps
        leak: negative slope for leaky ReLU
        seed: random seed for reproducibility
    """
    torch.manual_seed(seed)

    N = x.shape[1]

    if mask is None:
        # Default mask: lower-triangular with ones, so guaranteed full rank
        mask = torch.tril(torch.ones(N, N, device='cuda'))

    mask = mask.to(torch.float32)
    mask[1:,0] = 0 #Reserve Chromite its own Component

    # Initialize A with small random numbers where mask = 1
    A_params = nn.Parameter(torch.randn(N, N, device = 'cuda') * 0.1)

    # Keep diagonal entries nonzero for invertibility
    with torch.no_grad():
        for i in range(min(N, N)):
            if mask[i, i] == 1:
                A_params[i, i] = 1.0

    best_loss = np.inf
    
    optimizer = optim.Adam([A_params], lr=lr)
    activation = nn.LeakyReLU(negative_slope=leak)

    for step in range(steps):
        A = A_params * mask  # Apply mask
        pred = x @ A
        mse_loss = torch.mean((1-pred.sum(dim=1))**2)
        neg_penalty = torch.mean(activation(-pred))
        
        loss = mse_loss + neg_penalty  # Leaky ReLU loss on -pred

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Re-enforce diagonal non-zeros for invertibility
        with torch.no_grad():
            for i in range(N):
                if mask[i, i] == 1 and A_params[i, i].abs() < 1e-3:
                    A_params[i, i] = 1.0
                    

        if step % 500 == 0:
            print(f"lr = {lr}")
            print(f"Step {step}: Loss = {loss.item():.6f}")
            if loss.item() < best_loss:
                with torch.no_grad():
                    best_loss = loss.item()
                
            for i in range(N):
                for j in range(N):
                    if A_params[i,j] < 0.05: # Zero out small coefficients
                        with torch.no_grad():
                            A_params[i,j] = 0

        if (step+1) % 2500 == 0:
            lr /= 10
            optimizer = optim.Adam([A_params], lr=lr)
        
    return (A_params * mask).detach()



sp_present = np.where(Big_Table.table[:,component_indices['spinel']['mass (gm)']] != 0)[0]
sp_components = Big_Table.table[np.ix_(sp_present,np.arange(26,31))]

plt.hist(sp_components.sum(axis = 1))
plt.show()



Asp = solve_masked_A(torch.tensor(sp_components, requires_grad=True, device = 'cuda'), steps = 10000).detach().cpu().numpy()
AspInv = np.linalg.inv(Asp)


transformed = sp_components @ Asp
retransformed = transformed @ AspInv

print(np.array2string(Asp, formatter={'float_kind':lambda x: f"{x:6.2f}"}))
#print(transformed.sum(axis=1))
plt.hist((transformed).flatten())#, bins = [-0.2,0,0.2,0.4,0.6,0.8,1.0])
plt.title('BJT Remapped Sp Components')
plt.savefig('BJTSpRemapedComps.png')

plt.show()

print(f"MINIMUM REMAP: {np.min(transformed)}, MAXIMUM: {np.max(transformed)}")

plt.hist((transformed).sum(axis=1))
plt.title('Sum of BJT Remapped Sp Components (Offset by 1)')
plt.savefig('BJTSpRemapSums.png')
plt.show()
print((transformed > 1).sum() + (transformed < 0).sum())

plt.plot([-1.5,1.5],[-1.5,1.5], linestyle = '--', color = 'black')
plt.scatter(sp_components, retransformed, s= 5, alpha = 0.4)
#plt.scatter(sp_components[np.where(wrong_comps)[0]], retransformed[np.where(wrong_comps)[0]], s = 7, alpha = 0.7, marker = 'x', color = 'red')
plt.title('Spinel Remapping to MELTS Components')
plt.xlabel('Original MELTS Components')
plt.ylabel('Remapped MELTS Components')
plt.savefig('BJTrecastedSP.png')


In [ ]:
# Spinel Transformation Tests

Asp = np.array([[  1.00,   0.00,  0.00,  0.00,   0.00],
 [  0.00,   1.00,   0.00,   0.00,   0.00],
 [  0.00,   0.40,   0.60,   0.00,   0.00],
 [  0.00,  0.95,   0.00,   0.05,  0.00],
 [  0.00,   0.2,   0.00,   0.00,  0.8]])
AspInv = np.linalg.inv(Asp)


transformed = sp_components @ Asp
retransformed = transformed @ AspInv

print(np.array2string(Asp, formatter={'float_kind':lambda x: f"{x:6.2f}"}))
print(np.array2string(AspInv, formatter={'float_kind':lambda x: f"{x:6.2f}"}))

#print(transformed.sum(axis=1))
plt.hist((transformed).flatten())#, bins = [-0.2,0,0.2,0.4,0.6,0.8,1.0])
plt.title('BJT Remapped Sp Components')
plt.savefig('BJTSpRemapedComps.png')

plt.show()

print(f"MINIMUM REMAP: {np.min(transformed)}, MAXIMUM: {np.max(transformed)}")

plt.hist((transformed).sum(axis=1))
plt.title('Sum of BJT Remapped Sp Components (Offset by 1)')
plt.savefig('BJTSpRemapSums.png')
plt.show()
print((transformed > 1).sum() + (transformed < 0).sum())

plt.plot([-1.5,1.5],[-1.5,1.5], linestyle = '--', color = 'black')
plt.scatter(sp_components, retransformed, s= 5, alpha = 0.4)
#plt.scatter(sp_components[np.where(wrong_comps)[0]], retransformed[np.where(wrong_comps)[0]], s = 7, alpha = 0.7, marker = 'x', color = 'red')
plt.title('Spinel Remapping to MELTS Components')
plt.xlabel('Original MELTS Components')
plt.ylabel('Remapped MELTS Components')
plt.savefig('BJTrecastedSP.png')

In [ ]:
plt.hist([transformed[:,c] for c in np.arange(transformed.shape[1])],stacked=True,bins = np.linspace(0,1.05,21), label=np.arange(transformed.shape[1]))
plt.legend()
plt.title('Transformed Spinel Components')
plt.savefig('RecastSpHistogram.png')